In [1]:
import pandas as pd
import numpy as np
import urllib.request
import zipfile
import os
import timeit

# 1. Завантаження та розпакування датасету
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00235/household_power_consumption.zip"
zip_path = "household_power_consumption.zip"
txt_path = "household_power_consumption.txt"

if not os.path.exists(txt_path):
    print("Завантаження датасету (це може зайняти хвилину)...")
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall()
    print("Розпаковано!")
else:
    print("Файл вже існує. Переходимо до зчитування.")

# 2. Зчитування та очищення (Data Cleaning)
print("Зчитування даних у pandas dataframe...")
# Датасет розділений крапкою з комою (;), а пропуски в ньому це '?'
df = pd.read_csv(txt_path, sep=';', na_values=['?'], low_memory=False)

# Видаляємо всі рядки, де є пропущені значення (NaN)
df = df.dropna()

# Об'єднуємо Date та Time у зручний формат Datetime для подальшої роботи
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H:%M:%S')

print("Очищення завершено! Перші 5 рядків:")
display(df.head())

Завантаження датасету (це може зайняти хвилину)...
Розпаковано!
Зчитування даних у pandas dataframe...
Очищення завершено! Перші 5 рядків:


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Datetime
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0,2006-12-16 17:24:00
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0,2006-12-16 17:28:00


### Фільтрація даних: Потужність та Сила струму
* Обрати всі записи, у яких загальна активна споживана потужність перевищує 5 кВт.
* Обрати всі записи, у яких сила струму лежить в межах 19-20 А, для них виявити ті, у яких пральна машина та холодильних споживають більше, ніж бойлер та кондиціонер.
* Проаналізувати часові витрати на виконання процедур за допомогою модуля timeit.

In [2]:
import timeit

# Функція 1: Загальна активна споживана потужність > 5 кВт
def filter_high_power(data):
    # У датасеті Global_active_power вимірюється в кіловатах
    return data[data['Global_active_power'] > 5.0]

# Функція 2: Сила струму 19-20 А, Sub_metering_2 > Sub_metering_3
def filter_intensity_and_appliances(data):
    return data[(data['Global_intensity'] >= 19.0) & 
                (data['Global_intensity'] <= 20.0) & 
                (data['Sub_metering_2'] > data['Sub_metering_3'])]

# --- Виклик функцій та профілювання часу (timeit) ---

# Вимірюємо час для першої вибірки (запускаємо 1 раз)
time_power = timeit.timeit(lambda: filter_high_power(df), number=1)
res_power = filter_high_power(df)

# Вимірюємо час для другої вибірки
time_intensity = timeit.timeit(lambda: filter_intensity_and_appliances(df), number=1)
res_intensity = filter_intensity_and_appliances(df)

# --- Вивід результатів ---
print(f"--- Вибірка 1 (> 5 кВт) ---")
print(f"Знайдено записів: {len(res_power)}")
print(f"Час виконання pandas: {time_power:.5f} секунд")
display(res_power.head(3))

print(f"\n--- Вибірка 2 (19-20 А, Група 2 > Група 3) ---")
print(f"Знайдено записів: {len(res_intensity)}")
print(f"Час виконання pandas: {time_intensity:.5f} секунд")
display(res_intensity.head(3))

--- Вибірка 1 (> 5 кВт) ---
Знайдено записів: 17547
Час виконання pandas: 0.02039 секунд


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Datetime
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00



--- Вибірка 2 (19-20 А, Група 2 > Група 3) ---
Знайдено записів: 2509
Час виконання pandas: 0.03298 секунд


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Datetime
45,16/12/2006,18:09:00,4.464,0.136,234.66,19.0,0.0,37.0,16.0,2006-12-16 18:09:00
460,17/12/2006,01:04:00,4.582,0.258,238.08,19.6,0.0,13.0,0.0,2006-12-17 01:04:00
464,17/12/2006,01:08:00,4.618,0.104,239.61,19.6,0.0,27.0,0.0,2006-12-17 01:08:00


### Випадкова вибірка та складна фільтрація
* Обрати випадковим чином 500000 записів (без повторів елементів вибірки), для них обчислити середні величини усіх 3-х груп споживання електричної енергії.
* Обрати ті записи, які після 18-00 споживають понад 6 кВт за хвилину в середньому, серед відібраних визначити ті, у яких основне споживання припадає на групу 2 (вона є найбільшою).
* Потім обрати кожен третій результат із першої половини та кожен четвертий результат із другої половини.

In [3]:
import timeit
import pandas as pd

# Функція 3: Випадкова вибірка 500 000 записів та їх середнє
def random_sample_means(data):
    # Беремо 500 000 випадкових рядків без повторів (replace=False)
    sample = data.sample(n=500000, replace=False)
    # Обчислюємо середнє для трьох колонок
    return sample[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()

# Функція 4: Складна вибірка (після 18:00, > 6 кВт, Група 2 найбільша)
def filter_complex_evening(data):
    # Умова 1: час після 18:00 (у нас колонка Time - це рядок) та потужність > 6
    cond1 = (data['Time'] >= '18:00:00') & (data['Global_active_power'] > 6.0)
    
    # Умова 2: Група 2 більше за Групу 1 ТА Група 2 більше за Групу 3
    cond2 = (data['Sub_metering_2'] > data['Sub_metering_1']) & (data['Sub_metering_2'] > data['Sub_metering_3'])
    
    # Фільтруємо
    filtered = data[cond1 & cond2]
    
    # Розбиваємо результат на дві рівні половини
    half_index = len(filtered) // 2
    first_half = filtered.iloc[:half_index]
    second_half = filtered.iloc[half_index:]
    
    # Беремо кожен третій з першої (крок 3) та кожен четвертий з другої (крок 4)
    res1 = first_half.iloc[::3]
    res2 = second_half.iloc[::4]
    
    # Об'єднуємо назад
    return pd.concat([res1, res2])

# --- Профілювання часу (timeit) ---

time_sample = timeit.timeit(lambda: random_sample_means(df), number=1)
res_sample = random_sample_means(df)

time_complex = timeit.timeit(lambda: filter_complex_evening(df), number=1)
res_complex = filter_complex_evening(df)

# --- Вивід результатів ---
print(f"--- Вибірка 3 (Середнє для 500 000 випадкових записів) ---")
print(f"Час виконання: {time_sample:.5f} секунд")
print(res_sample)

print(f"\n--- Вибірка 4 (Після 18:00, складна фільтрація) ---")
print(f"Знайдено фінальних записів: {len(res_complex)}")
print(f"Час виконання: {time_complex:.5f} секунд")
display(res_complex.head(3))

--- Вибірка 3 (Середнє для 500 000 випадкових записів) ---
Час виконання: 0.34109 секунд
Sub_metering_1    1.131712
Sub_metering_2    1.297384
Sub_metering_3    6.467610
dtype: float64

--- Вибірка 4 (Після 18:00, складна фільтрація) ---
Знайдено фінальних записів: 310
Час виконання: 0.17639 секунд


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Datetime
41,16/12/2006,18:05:00,6.052,0.192,232.93,26.2,0.0,37.0,17.0,2006-12-16 18:05:00
44,16/12/2006,18:08:00,6.308,0.116,232.25,27.0,0.0,36.0,17.0,2006-12-16 18:08:00
17494,28/12/2006,20:58:00,6.386,0.374,236.63,27.0,1.0,36.0,17.0,2006-12-28 20:58:00


### Нормування, Стандартизація та Кореляція
* Пронормувати та стандартизувати вибраний датасет.
* Підрахувати коефіцієнт Пірсона та Спірмена для двох integer/real атрибутів. 
* Провести One Hot Encoding категоріального атрибута.

In [5]:
import pandas as pd

# Беремо випадкову підмножину даних (10 000 рядків) для швидких обчислень
df_subset = df.sample(n=10000, random_state=42).copy()

col1 = 'Global_active_power'
col2 = 'Global_intensity'

# 1. Нормування (Min-Max Scaling: від 0 до 1)
df_subset[f'{col1}_norm'] = (df_subset[col1] - df_subset[col1].min()) / (df_subset[col1].max() - df_subset[col1].min())

# 2. Стандартизація (Z-score: середнє = 0, стандартне відхилення = 1)
df_subset[f'{col2}_std'] = (df_subset[col2] - df_subset[col2].mean()) / df_subset[col2].std()

# 3. Підрахунок коефіцієнтів Пірсона та Спірмена для двох атрибутів
pearson_corr = df_subset[col1].corr(df_subset[col2], method='pearson')
spearman_corr = df_subset[col1].corr(df_subset[col2], method='spearman')

# 4. One Hot Encoding категоріального атрибута
# Створимо категоріальну колонку "День тижня" з Datetime
df_subset['Day_of_Week'] = df_subset['Datetime'].dt.day_name()

# Виконуємо One Hot Encoding за допомогою pd.get_dummies
df_ohe = pd.get_dummies(df_subset, columns=['Day_of_Week'], dtype=int)

# --- Вивід результатів ---
print(f"Коефіцієнт Пірсона між {col1} та {col2}: {pearson_corr:.4f}")
print(f"Коефіцієнт Спірмена між {col1} та {col2}: {spearman_corr:.4f}\n")

print("--- Результат нормалізації та стандартизації ---")
display(df_subset[[col1, f'{col1}_norm', col2, f'{col2}_std']].head())

print("\n--- Результат One Hot Encoding (колонки днів тижня) ---")
# Виводимо останні 7 колонок, щоб побачити результат OHE
display(df_ohe.iloc[:, -7:].head())

Коефіцієнт Пірсона між Global_active_power та Global_intensity: 0.9989
Коефіцієнт Спірмена між Global_active_power та Global_intensity: 0.9955

--- Результат нормалізації та стандартизації ---


,Global_active_power,Global_active_power_norm,Global_intensity,Global_intensity_std
1030580,1.502,0.168282,6.4,0.375171
1815,0.374,0.034980,1.8,-0.642488
1295977,0.620,0.064051,3.0,-0.377012
206669,0.280,0.023871,1.4,-0.730981
1048893,1.372,0.152919,5.6,0.198187



--- Результат One Hot Encoding (колонки днів тижня) ---


,Day_of_Week_Friday,Day_of_Week_Monday,Day_of_Week_Saturday,Day_of_Week_Sunday,Day_of_Week_Thursday,Day_of_Week_Tuesday,Day_of_Week_Wednesday
1030580,0,1,0,0,0,0,0
1815,0,0,0,1,0,0,0
1295977,0,0,0,0,0,0,1
206669,0,0,0,0,0,0,1
1048893,0,0,0,1,0,0,0
